# Understanding the Roman WFI Saturation Reference File 

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.1".

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.
    

## Introduction
The purpose of this notebook is to understand the the content and purpose of the **Saturation Threshold** (`SATURATION`) reference file.

The SATURATION reference provides a per-pixel saturation threshold (in DN).
During RomanCal processing, the SATURATION file is used in the `romancal.saturation.saturationStep()` step  to populate an array called `groupdq` in the science exposure. The step compares science data values against these thresholds and sets the SATURATED DQ flag (bit 1, value 2) when exceeded.

For more details, see the [romancal documentation](https://roman-pipeline.readthedocs.io/en/latest/roman/saturation/index.html) and [Rdox documentation](https://roman-docs.stsci.edu/data-handbook/roman-wfi-data-pipelines/exposure-level-pipeline#ExposureLevelPipeline-saturation) for SATURATION detection.


More details about this and other reference files can be found in the [Reference File Information](https://roman-pipeline.readthedocs.io/en/stable/roman/references_general/index.html)

### Local Run Settings

If you want to run the notebook in your local machine, refer to the information in [local installation](../../markdown/local-run.md) instructions before proceeding with the notebook. The instructions provide inportant information about setting up your environment and installing dependnecies.

## Imports
Libraries used:
- *astropy* for image normalization
- *copy* for making copies of Python objects
- *crds* for access to calibration reference files
- *matplotlib* and *mpl_toolkits* for plotting images
- *numpy* for array manipulation
- *roman_datamodels* for opening Roman WFI ASDF files
- *asdf* for opening Roman WFI ASDF files
- *os* for operating system functions

In [ ]:
import os
from astropy.visualization import simple_norm
import copy

import matplotlib.pyplot as plt
from matplotlib import colors, colormaps as cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import roman_datamodels as rdm

### The Calibration Reference Data System (CRDS)

 The reference files, developed and validated by STScI’s Science Operations Center, are continually updated as new WFI data become available. For more information about how CRDS works and how it assigns the most appropriate reference file for each calibration step, refer to the notebook [Understanding CRDS and How to Select Calibration Reference files](crds_reference_files.ipynb). 

**IMPORTANT NOTE:** Reference files are a work in progress and will be updated several times before Roman launch. If you notice irregularities or missing information, please understand that they may be a known issue. If you have questions, please contact the [Roman Help Desk](https://romanhelp.stsci.edu).

In [ ]:
import crds

Now let's dive into this reference file.

Let's check the environmental variables set for CRDS

In [ ]:
print(f"CRDS server location: {os.environ.get('CRDS_SERVER_URL')}")
print(f"CRDS context file: {os.environ.get('CRDS_CONTEXT')}")

If we want to change the context, we can do it in the next cell. In this case, we choose context `roman_0055.pmap`.

In [ ]:
os.environ['CRDS_CONTEXT']='roman_0055.pmap'

### Retrieving Reference Files

As you run the `RomanCal` pipeline, the most up-to-date reference files will be automatically selected for each step. However, if you would like to use a specific reference file, these can be retrieved through the `crds` Python API and the ELP run with those files (more on that later). Let's begin with how to access reference files from CRDS.

First, let's start by looking at the `crds.getrecommendations()` function. This function returns a dictionary of file names that match the criteria that you supply. Selection criteria are specified in a dictionary of key-value pairs. Each Roman WFI metadata keyword in the dictionary is all-caps and always begins with "ROMAN.META.". The remaining parts of the string correspond to the metadata keyword locations in the science data file schema. Different reference file types require different combinations of science metadata to match to the reference files. In general, all reference file types will require the instrument name ("INSTRUMENT.NAME") and start time ("EXPOSURE.START_TIME"). Most file types require the detector name ("INSTRUMENT.DETECTOR"), and some file types require the exposure type ("EXPOSURE.TYPE") or optical element ("INSTRUMENT.OPTICAL_ELEMENT").

For the saturation files in particular, the required keywords are:
- saturation
    - ROMAN.META.INSTRUMENT.NAME
    - ROMAN.META.INSTRUMENT.DETECTOR
    - ROMAN.META.EXPOSURE.START_TIME

These keywords may be combined into a single dictionary to find multiple reference file types using `crds.getreferences()`. For example, if you would like to find the name of the dark and flat reference files used by the pipeline, you could run the following example:

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getrecommendations(meta, reftypes=['saturation'], observatory='roman')

The `ref_files` variable now contains a dictionary for the MASK reference file. This is the reference file that correspond to a science observation taken at midnight UTC on January 1, 2026 in the WFI imaging mode with optical element F158 and detector WFI01. Let's take a look at the names of the files CRDS returned:

In [ ]:
ref_files

We can also use `crds.getreferences()` to accomplish the same thing; however, `getreferences()` goes one step further beyond `getrecommendations()` and will download the reference files if they are not already in your local cache. Using the same example as above:

In [ ]:
meta = {'ROMAN.META.INSTRUMENT.NAME': 'WFI',
        'ROMAN.META.INSTRUMENT.DETECTOR': 'WFI01',
        'ROMAN.META.EXPOSURE.START_TIME': '2026-01-01 00:00:00'
       }

ref_files = crds.getreferences(meta, reftypes=['saturation'], observatory='roman')

Once again we can examine the output of `ref_files`:

In [ ]:
ref_files

### Examining Reference Files

Reference files use `roman_datamodels` just like WFI science data products and can be accessed in the same way (see the tutorial [Working with ASDF](../working_with_asdf/working_with_asdf.ipynb) for more information). Let's take a closer look at the files we retrieved from our `crds.getreferences()` example starting with the mask file:

In [ ]:
saturation = rdm.open(ref_files['saturation'])
saturation.info()

We see that the SATURATION file contains metadata and a `data` and `dq` arrays. For this file, the `data` array contains the Saturation Thresholds. 

Let's take a look at the saturation data for this detector. First, lets check the shape of the data array:

In [ ]:
print("saturation.data shape:", saturation.data.shape)
print("saturation.dq shape:", saturation.dq.shape)

Now lets get some basic statistics on the cube (or a representative slice)

In [ ]:
print("sat.data shape:", saturation.data.shape)
print("sat.dq shape:", saturation.dq.shape)

# Quick stats on saturation thresholds
data = saturation.data
print("\nSaturation threshold stats:")
print("  Min:", data.min(), "Max:", data.max())
print("  Mean:", data.mean(), "Median:", np.median(data))
print("  Std:", data.std())

# DQ stats 
dq = saturation.dq
total = dq.size
flagged = np.sum(dq > 0)
print(f"\nDQ: {flagged:,} / {total:,} pixels flagged ({flagged/total*100:.3f}%)")

Now lets get a histobran of the saturation threshold values

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(saturation.data.flatten(), bins=100, log=True)
plt.xlabel('Saturation Threshold (DN)')
plt.ylabel('Number of Pixels (log scale)')
plt.title('Distribution of Saturation Thresholds')
plt.grid(True, alpha=0.3)
plt.show()

Now, let's plot two versions of the file, one with ethe threshold map and another flattened version with DQ map:

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 7))

# Saturation threshold map
norm = simple_norm(saturation.data, stretch='sqrt', percent=99)
im0 = axs[0].imshow(saturation.data, cmap='viridis', norm=norm, origin='lower')
axs[0].set_title('Saturation Threshold (DN)')
divider = make_axes_locatable(axs[0])
cax = divider.append_axes("right", size="5%", pad=0.05)
fig.colorbar(im0, cax=cax, label='DN')

# DQ map
axs[1].imshow(np.bool_(saturation.dq), cmap='binary_r', origin='lower')
axs[1].set_title('Saturation Reference DQ Flags')

for ax in axs:
    ax.set_xlabel('Science X (pixels)')
    ax.set_ylabel('Science Y (pixels)')

plt.tight_layout()
plt.show()

## About this Notebook
**Author:** T. Desjardins. & R. Diaz

**Updated On:** 2026-07-06

<table width="100%" style="border:none; border-collapse:collapse;">

  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>